# 10 · Explanations for the app (XAI), with a simple attribution check

## Purpose
The app shows, next to each diagnosis, **what the models attended to**: for the CNN-6 a channels × order-band map, and for
the logistic-regression reference the R1 features that pushed it towards its class. This notebook shows those
explanations on a few development recordings, one imbalance ↔ vertical-misalignment confusion case, and a simple
occlusion check of whether the highlighted order bands matter to the model.

**Wording.** Attribution shows what the model attended to; it is **not proof of the fault's location**, and a predicted
fault location is according to the dataset labels (CLAUDE.md §3). The app is a research demonstration with documented
limitations; these are development recordings on one partition, and reserved tests stay sealed.

## Inputs
The verified model bundle `artifacts/bundle/` (Stage ML-4: trained on closed train ∩ development pool, calibrated on
closed val ∩ development pool), `ml.inference.diagnose`, `ml.xai`; for the confusion case the saved Stage 3C LOCO CNN-6
(imbalance withheld, fold 0, seed 42).

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the settings and the verified bundle, and picks the examples: for every class the first calibration recording (closed val ∩ development pool) by name. Calibration recordings set thresholds but are not training data.</div>

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from ml.config import load_config
from ml.inference import diagnose, get_bundle
from ml.labels import CLASS10
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR
from ml.splits import development_pool
from ml.xai import NOTE

cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
FIG = SYSTEM_CODE_DIR / "results" / "figures"
plt.rcParams.update({"figure.dpi": 110})
bundle = get_bundle()
pool = development_pool(meta, cfg.development.allowed_closed, cfg.development.reserved_columns)
cal = pool[pool.split_closed == "val"].sort_values("rec_id")
examples = cal.groupby("class10").rec_id.first().reindex(CLASS10)
print(NOTE)
examples.to_frame("example recording")

## 1. Diagnoses and explanations per class

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Runs diagnose() on each example and tabulates the decision, the CNN's top channels and order bands (input × gradient shares) and the logistic regression's top feature.</div>

In [ ]:
res = {c: diagnose(r) for c, r in examples.items()}
rows = []
for c, r in res.items():
    x = r["xai"]
    rows.append({"true class": c, "CNN top class": r["diagnosis"]["top_class"], "decision": r["diagnosis"]["display_label"],
                 "top channels": ", ".join(f"{t['channel']} ({t['share']:.2f})" for t in x["cnn"]["top_channels"]),
                 "top order bands": ", ".join(f"{b['orders'][0]:.2f}–{b['orders'][1]:.2f}" for b in x["cnn"]["top_order_bands"]),
                 "LR class": x["lr"]["class"], "LR top feature": x["lr"]["top_features"][0]["feature"]})
pd.DataFrame(rows).set_index("true class")

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws the coarse input × gradient maps (six vibration channels × 32 order groups on the log order axis) for the ten examples; brighter = more attribution for the CNN's top class.</div>

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 6), sharey=True)
for ax, (c, r) in zip(axes.ravel(), res.items()):
    h = r["xai"]["cnn"]["heat_map"]
    ax.imshow(np.array(h["values"]), aspect="auto", cmap="magma", vmin=0, vmax=1)
    ticks = [0, 8, 16, 24, 31]
    ax.set_xticks(ticks, [f"{h['order_groups'][i][0]:.1f}" for i in ticks], fontsize=7)
    ax.set_yticks(range(6), h["channels"], fontsize=7)
    ax.set_title(f"{c}\n→ {r['diagnosis']['top_class']}", fontsize=8)
    ax.set_xlabel("order (group start)", fontsize=7)
fig.suptitle("CNN-6 input × gradient: what the model attended to (not proof of fault location)")
fig.tight_layout()
fig.savefig(FIG / "10_xai_heatmaps.png")

## 2. A confusion case: withheld imbalance called vertical misalignment
In the Stage 3C LOCO run with imbalance withheld (fold 0, seed 42), 139 of 144 withheld imbalance instances were assigned
vertical misalignment over the three folds. Below, one such recording is explained with **that LOCO model** (which never
saw imbalance) for its predicted class, next to the bundle CNN (which was trained with both classes; this recording is in
the bundle's training set, so its row is not an evaluation). Descriptive only; no causal claim about why the two classes
are confused.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the LOCO CNN-6 (imbalance withheld, fold 0) with its own normalisation, computes the representation of one withheld imbalance recording that it assigned to vertical misalignment, and draws the attribution maps of the LOCO model and of the bundle CNN side by side.</div>

In [ ]:
from ml.config import CnnConfig
from ml.models.cnn import ScratchCNN
from ml.paths import data_dir, results_dir
from ml.spectro import order_axis, stacked_representation
from ml.train import normalise
from ml.xai import cnn_attribution

rid = "imbalance__15g__13.312"
known = [c for c in CLASS10 if c != "imbalance"]
loco = ScratchCNN(6, 9, CnnConfig(**bundle.model_config["cnn"]))
loco.load_state_dict(torch.load(SYSTEM_CODE_DIR / "artifacts" / "loco" / "cnn6_w-imbalance_s42_loco" / "fold0.pt", weights_only=True))
loco.eval()
nl = json.loads((results_dir() / "loco" / "cnn6_w-imbalance_s42_loco" / "fold0_norm.json").read_text())
x = np.load(data_dir() / "raw_npy" / f"{rid}.npy")
rep = stacked_representation(x, float(meta.set_index("rec_id").loc[rid, "fr_measured_hz"]), cfg.preprocess,
                             cfg.representation.leading_candidate)
xr = torch.from_numpy(rep[:, :6]).to(torch.float32)
xl = normalise(xr, torch.tensor(nl["mean"], dtype=torch.float32), torch.tensor(nl["std"], dtype=torch.float32))
with torch.no_grad():
    p = torch.softmax(loco(xl)[0], dim=1).mean(0).numpy()
pred = known[int(p.argmax())]
att_loco = cnn_attribution(loco, xl, known.index(pred), order_axis(cfg.preprocess))
r_b = diagnose(rid)
print(f"LOCO model (imbalance withheld): predicted {pred} (mean window probability {p.max():.3f})")
print(f"bundle CNN: {r_b['diagnosis']['top_class']} → {r_b['diagnosis']['display_label']}")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4), sharey=True)
for ax, (title, h) in zip(axes, ((f"LOCO CNN-6 → {pred}", att_loco["heat_map"]),
                                  (f"bundle CNN-6 → {r_b['diagnosis']['top_class']}", r_b["xai"]["cnn"]["heat_map"]))):
    ax.imshow(np.array(h["values"]), aspect="auto", cmap="magma", vmin=0, vmax=1)
    ticks = [0, 8, 16, 24, 31]
    ax.set_xticks(ticks, [f"{h['order_groups'][i][0]:.1f}" for i in ticks], fontsize=7)
    ax.set_yticks(range(6), h["channels"], fontsize=7)
    ax.set_title(title, fontsize=9)
fig.suptitle(f"{rid}: what each model attended to (descriptive; not a cause)")
fig.tight_layout()
fig.savefig(FIG / "10_xai_confusion_case.png")
pd.DataFrame({"LOCO → " + pred: [f"{t['channel']} ({t['share']:.2f})" for t in att_loco["top_channels"]],
              "bundle → " + r_b["diagnosis"]["top_class"]: [f"{t['channel']} ({t['share']:.2f})" for t in r_b["xai"]["cnn"]["top_channels"]]})

## 3. Occlusion check (descriptive)
If the highlighted order bands matter to the CNN, setting them to the channel mean (0 in normalised units) should lower
the top-class probability more than occluding a random band of the same width. One band (4 order bins, all channels) per
example; the random band is drawn with a fixed seed. Because the top-class probabilities are saturated near 1, the check
also reports the top-class **logit margin** (top logit minus the largest other logit, mean over windows), which does not
saturate. A sanity check of the attribution, not a validation.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> For each example, occludes the top-attributed order group and, separately, a random group of the same width, and records the change in the CNN's top-class probability and in its top-class logit margin (means over windows).</div>

In [ ]:
rng = np.random.default_rng(0)
nc = bundle.normalisation["cnn6"]
rows = []
for c, rid_ in examples.items():
    x = np.load(data_dir() / "raw_npy" / f"{rid_}.npy")
    rep = stacked_representation(x, float(meta.set_index("rec_id").loc[rid_, "fr_measured_hz"]), cfg.preprocess,
                                 cfg.representation.leading_candidate)
    xn = normalise(torch.from_numpy(rep[:, :6]).to(torch.float32), torch.tensor(nc["mean"], dtype=torch.float32),
                   torch.tensor(nc["std"], dtype=torch.float32))
    k = CLASS10.index(res[c]["diagnosis"]["top_class"])
    heat = np.array(res[c]["xai"]["cnn"]["heat_map"]["values"]).sum(axis=0)
    top_g, rand_g = int(heat.argmax()), int(rng.integers(0, 32))

    def run(g=None):
        z = xn.clone()
        if g is not None:
            z[:, :, 4 * g: 4 * g + 4, :] = 0.0
        with torch.no_grad():
            lg = bundle.cnn(z)[0]
        other = torch.cat([lg[:, :k], lg[:, k + 1:]], dim=1).max(dim=1).values
        return float(torch.softmax(lg, dim=1)[:, k].mean()), float((lg[:, k] - other).mean())

    (p0, m0), (p1, m1), (p2, m2) = run(), run(top_g), run(rand_g)
    rows.append({"class": c, "probability": p0, "prob. drop, top band": p0 - p1, "prob. drop, random band": p0 - p2,
                 "logit margin": m0, "margin drop, top band": m0 - m1, "margin drop, random band": m0 - m2})
occ = pd.DataFrame(rows).set_index("class")
display(occ.round(4))
occ.drop(columns=["probability", "logit margin"]).mean().round(4)

## Findings
Development recordings on one partition; reserved tests sealed. Attribution shows what the model attended to; it is not
proof of the fault's location.
- **The CNN attribution is diffuse.** For the ten calibration examples, the top channel holds 0.21–0.30 of the input ×
  gradient attribution and each top order band only about 5–7 %; the tangential and axial channels lead in most
  examples, with the high-order bands (> 75 orders) often on top.
- **The occlusion check does not confirm single-band importance.** Occluding the top-attributed order band lowers the
  top-class logit margin by 0.22 on average, a random band of the same width by 0.26; top-class probabilities are
  saturated (≥ 0.999), so they barely move either way. The maps are therefore a coarse description of what the model
  weighted, not evidence that a particular order band drives the decision.
- **LR attribution** names concrete R1 features (e.g. imbalance → f_under_tangential_peak and overhang-axial order lines;
  vertical misalignment → f_over_tangential_o4); these are coefficient × transformed-value contributions for LR's own
  class, not causes.
- **Confusion case (descriptive):** the Stage 3C LOCO CNN-6 with imbalance withheld assigns `imbalance__15g__13.312` to
  vertical misalignment (0.988); its attribution leads with over_axial (0.25), while the bundle CNN (trained with both
  classes; this recording is in its training set, so not an evaluation) predicts imbalance and leads with
  over_tangential. No causal claim is made about why the two classes are confused.
- **Decisions on the examples:** the normal example is "healthy — caution" (a healthy-baseline check disagrees) and the
  imbalance example is "further inspection needed"; the other eight are the correct known fault. These are calibration
  recordings (used for thresholds), so this is not an evaluation.


## Outputs written
- `results/figures/10_xai_heatmaps.png`, `10_xai_confusion_case.png`